In [2]:
import requests
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

API_URL = "https://datasets-server.huggingface.co/filter"

session = requests.Session()

retry = Retry(
    total=6,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
    respect_retry_after_header=True,
)

session.mount("https://", HTTPAdapter(max_retries=retry))

base_params = {
    "dataset": "GeorgiaCh96/WESAD_raw_data",
    "config": "default",
    "split": "train",
    "where": '"subject id"=2',  # Correct column name
}

rows = []
offset = 0
total = None

while total is None or offset < total:
    params = {
        **base_params,
        "offset": offset,
        "length": 100,
    }

    response = session.get(API_URL, params=params, timeout=60)
    response.raise_for_status()
    result = response.json()

    if result.get("partial", False):
        raise RuntimeError(
            "The API filtered only part of the dataset."
        )

    if total is None:
        total = result["num_rows_total"]
        print(f"Participant S2: {total:,} rows")

    page = [item["row"] for item in result["rows"]]

    if not page:
        raise RuntimeError(
            f"No rows returned at offset {offset}."
        )

    rows.extend(page)
    offset += len(page)

    print(f"Downloaded {offset:,}/{total:,} rows")

df = pd.DataFrame(rows)
df.to_csv("WESAD_S2_features.csv", index=False)

print("\nSaved successfully!")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head())

Participant S2: 11,973 rows
Downloaded 100/11,973 rows
Downloaded 200/11,973 rows
Downloaded 300/11,973 rows
Downloaded 400/11,973 rows
Downloaded 500/11,973 rows
Downloaded 600/11,973 rows
Downloaded 700/11,973 rows
Downloaded 800/11,973 rows
Downloaded 900/11,973 rows
Downloaded 1,000/11,973 rows
Downloaded 1,100/11,973 rows
Downloaded 1,200/11,973 rows
Downloaded 1,300/11,973 rows
Downloaded 1,400/11,973 rows
Downloaded 1,500/11,973 rows
Downloaded 1,600/11,973 rows
Downloaded 1,700/11,973 rows
Downloaded 1,800/11,973 rows
Downloaded 1,900/11,973 rows
Downloaded 2,000/11,973 rows
Downloaded 2,100/11,973 rows
Downloaded 2,200/11,973 rows
Downloaded 2,300/11,973 rows
Downloaded 2,400/11,973 rows
Downloaded 2,500/11,973 rows
Downloaded 2,600/11,973 rows
Downloaded 2,700/11,973 rows
Downloaded 2,800/11,973 rows
Downloaded 2,900/11,973 rows
Downloaded 3,000/11,973 rows
Downloaded 3,100/11,973 rows
Downloaded 3,200/11,973 rows
Downloaded 3,300/11,973 rows
Downloaded 3,400/11,973 rows
Down

In [3]:
from pathlib import Path

file = Path("WESAD_S2_features.csv")

print(f"CSV size: {file.stat().st_size / 1024**2:.2f} MB")

CSV size: 12.89 MB
